In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
import torch

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration
MODEL_NAME = "microsoft/codebert-base"
NUM_LABELS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 3

class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def load_and_preprocess_data(base_path, tokenizer, folder_name):
    codes = []
    labels = []
    for label_dir in ["Label_0", "Label_1"]:
        current_path = os.path.join(base_path, folder_name, label_dir)
        if not os.path.exists(current_path):
            print(f"Warning: Directory {current_path} not found. Skipping.")
            continue
        for filename in os.listdir(current_path):
            if filename.endswith(".txt"):
                filepath = os.path.join(current_path, filename)
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    codes.append(f.read())
                labels.append(0 if label_dir == "Label_0" else 1)

    # Tokenize the codes
    encodings = tokenizer(codes, truncation=True, padding=True, max_length=512)
    return CodeDataset(encodings, labels)



In [3]:
from transformers import RobertaModel


base_text_path = "../Text_Files/"

# Load tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.eval()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [4]:
# Load and preprocess training data
print("Loading training & validation data...")
train_dataset = load_and_preprocess_data(base_text_path, tokenizer, "Train")

Loading training & validation data...


In [5]:
# Training arguments
training_args = TrainingArguments(
    output_dir="../checkpoints/codebert_only",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="no",
    save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
)

In [6]:
# Trainer 
trainer = Trainer( model=model, args=training_args, train_dataset=train_dataset, ) 
print("Training model...") 
trainer.train()

Training model...


Step,Training Loss
10,0.761800
20,0.723100
30,0.717700
40,0.683400
50,0.710000
60,0.695000
70,0.709300
80,0.677800
90,0.674300
100,0.681500


TrainOutput(global_step=2322, training_loss=0.30016065429186994, metrics={'train_runtime': 1346.8918, 'train_samples_per_second': 13.787, 'train_steps_per_second': 1.724, 'total_flos': 4885972298035200.0, 'train_loss': 0.30016065429186994, 'epoch': 3.0})

In [8]:
import os
current_dir = os.getcwd()
print(current_dir)

/home/info-sec-lab/BTP/SO/experiments


In [11]:
model_path = "../checkpoints/codebert_only/checkpoint-2322"
model = RobertaForSequenceClassification.from_pretrained(model_path)

# Load tokenizer from the original pre-trained model (not from checkpoint)
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")  # or whatever base model you used

# Create trainer with the loaded model
trainer = Trainer(model=model)

## 🧪 Testing on Checkpoint 1 — `CodeBERT`


In [12]:
# Evaluate on test datasets
print("Evaluating on test datasets...")
for i in range(10):
    test_folder = f"Test_{i}"
    print(f"Loading test data for {test_folder}...")
    test_dataset = load_and_preprocess_data(base_text_path, tokenizer, test_folder)
    if len(test_dataset) > 0:
        predictions = trainer.predict(test_dataset)
        # Process predictions to get labels
        predicted_labels = predictions.predictions.argmax(axis=1)
        true_labels = test_dataset.labels

        from sklearn.metrics import accuracy_score, precision_recall_fscore_support
        accuracy = accuracy_score(true_labels, predicted_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='binary')

        print(f"Results for {test_folder}:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    else:
        print(f"No data found for {test_folder}. Skipping evaluation.")


Evaluating on test datasets...
Loading test data for Test_0...


Results for Test_0:
  Accuracy: 0.7874
  Precision: 0.8229
  Recall: 0.7325
  F1-Score: 0.7751
Loading test data for Test_1...


Results for Test_1:
  Accuracy: 0.7754
  Precision: 0.8013
  Recall: 0.7325
  F1-Score: 0.7654
Loading test data for Test_2...


Results for Test_2:
  Accuracy: 0.7399
  Precision: 0.7384
  Recall: 0.7325
  F1-Score: 0.7355
Loading test data for Test_3...


Results for Test_3:
  Accuracy: 0.7465
  Precision: 0.7536
  Recall: 0.7325
  F1-Score: 0.7429
Loading test data for Test_4...


...

Results for Test_5:
  Accuracy: 0.8214
  Precision: 0.8908
  Recall: 0.7325
  F1-Score: 0.8039
Loading test data for Test_6...


Results for Test_6:
  Accuracy: 0.7475
  Precision: 0.7551
  Recall: 0.7325
  F1-Score: 0.7437
Loading test data for Test_7...


Results for Test_7:
  Accuracy: 0.6916
  Precision: 0.6771
  Recall: 0.7325
  F1-Score: 0.7037
Loading test data for Test_8...


Results for Test_8:
  Accuracy: 0.7475
  Precision: 0.7551
  Recall: 0.7325
  F1-Score: 0.7437
Loading test data for Test_9...


Results for Test_9:
  Accuracy: 0.6916
  Precision: 0.6771
  Recall: 0.7325
  F1-Score: 0.7037
